# Sensibilidade dos filtros do grafo `user_game`

Este notebook testa configuracoes alternativas de filtro para o grafo bipartido usuario-jogo, usando como base o dataset processado `data/processed/user_game_edges.csv`.

A motivacao e reduzir o tamanho do grafo e, principalmente, o custo combinatorio da futura projecao jogo-jogo, sem alterar a coerencia metodologica do projeto.

## Criterio de avaliacao

Foram avaliados dois controles:

- `max_user_games`: limite superior de jogos por usuario. Esse filtro controla diretamente usuarios que induzem muitas coocorrencias na projecao.
- `min_game_users`: minimo de usuarios por jogo apos o filtro de usuarios. Esse filtro remove jogos muito perifericos e aumenta estabilidade estatistica.

A metrica mais importante para a etapa de projecao e `projection_pairs_from_users`, calculada como `sum(C(d_u, 2))`, onde `d_u` e o grau do usuario apos os filtros. Ela estima quantos pares jogo-jogo os usuarios induziriam na projecao comportamental.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = Path('..').resolve()
SENSITIVITY_CSV = BASE_DIR / 'data/processed/user_game_filter_sensitivity.csv'

df = pd.read_csv(SENSITIVITY_CSV)
df.head()

In [ ]:
cols = [
    'max_user_games',
    'min_game_users',
    'n_users',
    'n_games',
    'n_edges',
    'edges_preserved_pct',
    'projection_pairs_preserved_pct',
    'graphml_estimated_gib',
    'users_degree_1_after_game_filter',
]

df[cols].sort_values(['max_user_games', 'min_game_users'])

## Leitura inicial

O limite superior de jogos por usuario e o controle mais efetivo. Reduzir `max_user_games` de `100` para `30` preserva cerca de `87%` das arestas, mas reduz o custo combinatorio estimado da projecao para cerca de `49%` do valor original. Isso e uma troca metodologicamente interessante: perde-se relativamente pouco em cobertura de interacoes, mas corta-se aproximadamente metade da explosao de pares jogo-jogo.

O filtro `min_game_users` reduz fortemente o numero de jogos, mas quase nao reduz o numero de arestas quando aplicado depois do filtro de usuarios. Isso indica que os jogos removidos sao muito perifericos: muitos jogos desaparecem, mas eles concentram poucas interacoes. Portanto, esse filtro e mais um controle de estabilidade e ruido do que um controle primario de tamanho.

In [ ]:
pareto = df[
    (df['edges_preserved_pct'] >= 80)
    & (df['projection_pairs_preserved_pct'] <= 60)
].copy()

pareto[cols].sort_values(['max_user_games', 'min_game_users'])

In [ ]:
def heatmap(metric: str, title: str, fmt: str = '.1f'):
    pivot = df.pivot(index='max_user_games', columns='min_game_users', values=metric)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    im = ax.imshow(pivot.values, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('Minimo de usuarios por jogo')
    ax.set_ylabel('Maximo de jogos por usuario')
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, format(pivot.values[i, j], fmt), ha='center', va='center', color='white', fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    return fig, ax

heatmap('edges_preserved_pct', 'Arestas preservadas (%)')
plt.show()

heatmap('projection_pairs_preserved_pct', 'Custo combinatorio da projecao preservado (%)')
plt.show()

heatmap('graphml_estimated_gib', 'Tamanho estimado do GraphML (GiB)', fmt='.2f')
plt.show()

## Configuracoes candidatas

As configuracoes mais defensaveis nesta rodada sao:

- `max_user_games = 30`, `min_game_users = 10`: preserva `86,97%` das arestas, `98,59%` dos usuarios e `63,72%` dos jogos, enquanto reduz o custo da projecao para `49,16%` do original.
- `max_user_games = 30`, `min_game_users = 20`: preserva `86,73%` das arestas e `48,85%` dos jogos, com custo de projecao em `48,81%`. E um pouco mais agressivo contra jogos perifericos.
- `max_user_games = 20`, `min_game_users = 10`: reduz o custo da projecao para `35,12%`, mas preserva apenas `79,23%` das arestas. E uma alternativa mais forte se a projecao ainda ficar grande demais.

A configuracao `max_user_games = 50` parece menos interessante: ainda preserva mais de `94%` das arestas, mas mantem cerca de `70%` do custo combinatorio da projecao. Ela reduz pouco justamente o problema principal.

In [ ]:
recommended = df[
    ((df.max_user_games == 30) & (df.min_game_users.isin([10, 20])))
    | ((df.max_user_games == 20) & (df.min_game_users == 10))
    | ((df.max_user_games == 50) & (df.min_game_users == 10))
]
recommended[cols].sort_values(['max_user_games', 'min_game_users'])

## Recomendacao preliminar

A melhor configuracao preliminar e `2 <= jogos_por_usuario <= 30` combinada com `usuarios_por_jogo >= 10`.

Justificativa:

- reduz aproximadamente metade do custo combinatorio da projecao;
- preserva a imensa maioria dos usuarios;
- preserva quase `87%` das arestas;
- remove jogos muito perifericos, mas sem cortar volume relevante de interacoes;
- mantem uma rede ainda grande o suficiente para analise comportamental robusta.

A configuracao com `usuarios_por_jogo >= 20` tambem e defensavel se o objetivo for priorizar estabilidade estatistica sobre cobertura de nicho. A escolha final deve depender do comportamento da projecao jogo-jogo: se a rede projetada continuar densa demais, o limiar `20` passa a ser mais atrativo.

## Observacao metodologica

Reduzir o grafo nao deve ser apresentado como uma decisao puramente computacional. A justificativa correta e metodologica: usuarios com muitos jogos tem peso combinatorio desproporcional na projecao, e jogos com pouquissimos usuarios tendem a gerar relacoes pouco estaveis. Assim, filtros mais restritivos podem melhorar a interpretabilidade da rede projetada, desde que a preservacao de usuarios, arestas e estrutura global continue alta.